# ✅ Drop a catalog (with all schemas + tables)

## What this does
* Removes the catalog

* CASCADE also removes all schemas and objects inside it

* Managed tables are soft-deleted for 7 days before permanent purge

* External tables only lose metadata; underlying data stays

In [0]:
%sql
DROP CATALOG IF EXISTS jim_bakery CASCADE;

# 🔄 If you want to recreate it afterward

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS jim_bakery
  COMMENT 'Bakery data platform catalog';

# 🧁 2. Create Schemas

* **SKU** stands for Stock Keeping Unit — it’s the unique identifier a business uses to track a specific sellable item.

In [0]:
%sql
-- Customer domain
CREATE SCHEMA IF NOT EXISTS jim_bakery.customers
  COMMENT 'Customer identity, orders, loyalty';

-- Bakery operations domain
CREATE SCHEMA IF NOT EXISTS jim_bakery.bakery
  COMMENT 'Operational bakery schema: batches, inventory, ovens';

-- Product domain
CREATE SCHEMA IF NOT EXISTS jim_bakery.bake_goods
  COMMENT 'Product catalog, SKUs, pricing';

# 🍞 3. Bakery Schema Tables

In [0]:
%sql
USE CATALOG jim_bakery;
USE SCHEMA bakery;

-- Ingredient Inventory
CREATE TABLE IF NOT EXISTS ingredient_inventory (
  ingredient_id STRING,
  ingredient_name STRING,
  unit STRING,
  quantity DOUBLE,
  reorder_level DOUBLE,
  supplier STRING,
  last_updated TIMESTAMP
);

-- Batch Log
CREATE TABLE IF NOT EXISTS batch_log (
  batch_id STRING,
  product_id STRING,
  start_time TIMESTAMP,
  end_time TIMESTAMP,
  quantity INT,
  baker STRING,
  oven_id STRING,
  status STRING
);

-- Recipe Master
CREATE TABLE IF NOT EXISTS recipe_master (
  recipe_id STRING,
  product_id STRING,
  ingredient_id STRING,
  amount DOUBLE,
  unit STRING
);

-- Oven Utilization
CREATE TABLE IF NOT EXISTS oven_utilization (
  oven_id STRING,
  batch_id STRING,
  start_time TIMESTAMP,
  end_time TIMESTAMP,
  temperature DOUBLE,
  notes STRING
);

-- Waste Log
CREATE TABLE IF NOT EXISTS waste_log (
  waste_id STRING,
  batch_id STRING,
  ingredient_id STRING,
  quantity DOUBLE,
  reason STRING,
  timestamp TIMESTAMP
);

-- Daily Production Summary
CREATE TABLE IF NOT EXISTS daily_production (
  production_date DATE,
  product_id STRING,
  total_batches INT,
  total_quantity INT,
  failed_batches INT,
  total_waste DOUBLE
);

# 🧁 4. bake_goods Schema Tables

In [0]:
%sql
USE CATALOG jim_bakery;
USE SCHEMA bake_goods;

-- Product Catalog
CREATE TABLE IF NOT EXISTS product_catalog (
  product_id STRING,
  product_name STRING,
  category STRING,
  description STRING,
  price DOUBLE,
  active BOOLEAN
);

-- SKU Metadata
CREATE TABLE IF NOT EXISTS sku_metadata (
  sku STRING,
  product_id STRING,
  size STRING,
  weight DOUBLE,
  packaging STRING
);


# 👥 5. customers Schema Tables

In [0]:
%sql
USE CATALOG jim_bakery;
USE SCHEMA customers;
-- Customer Profile
CREATE TABLE IF NOT EXISTS customer_profile (
  customer_id STRING,
  name STRING,
  email STRING,
  phone STRING,
  loyalty_tier STRING,
  created_at TIMESTAMP
);

-- Orders
CREATE TABLE IF NOT EXISTS orders (
  order_id STRING,
  customer_id STRING,
  order_date TIMESTAMP,
  total_amount DOUBLE,
  status STRING
);

-- Order Items
CREATE TABLE IF NOT EXISTS order_items (
  order_id STRING,
  sku STRING,
  quantity INT,
  price DOUBLE
);


# 🔗 Lineage Summary
* customers → orders → order_items → bake_goods

* bake_goods → recipe_master → batch_log → daily_production

* ingredient_inventory → recipe_master → batch_log → waste_log

This gives you full traceability from ingredient → batch → product → customer order.

# 🍞 Bakery Schema — Sample Data 

In [0]:
%sql
USE CATALOG jim_bakery;
USE SCHEMA bakery;

-- Ingredient Inventory
INSERT INTO ingredient_inventory VALUES
('ING-001','Flour','kg',120,50,'MillCo',current_timestamp()),
('ING-002','Sugar','kg',80,30,'SweetCorp',current_timestamp()),
('ING-003','Butter','kg',40,20,'DairyBest',current_timestamp()),
('ING-004','Eggs','units',300,100,'FarmFresh',current_timestamp());

-- Recipe Master
INSERT INTO recipe_master VALUES
('REC-001','PROD-001','ING-001',1.5,'kg'),
('REC-001','PROD-001','ING-002',0.2,'kg'),
('REC-001','PROD-001','ING-004',4,'units'),
('REC-002','PROD-002','ING-001',1.0,'kg'),
('REC-002','PROD-002','ING-003',0.5,'kg');

-- Batch Log
INSERT INTO batch_log VALUES
('BATCH-1001','PROD-001',current_timestamp()-INTERVAL 2 HOURS,current_timestamp()-INTERVAL 1 HOURS,120,'John','OVEN-1','completed'),
('BATCH-1002','PROD-002',current_timestamp()-INTERVAL 3 HOURS,current_timestamp()-INTERVAL 2 HOURS,80,'Maria','OVEN-2','completed'),
('BATCH-1003','PROD-001',current_timestamp()-INTERVAL 1 HOURS,current_timestamp(),0,'John','OVEN-1','failed');

-- Oven Utilization
INSERT INTO oven_utilization VALUES
('OVEN-1','BATCH-1001',current_timestamp()-INTERVAL 2 HOURS,current_timestamp()-INTERVAL 1 HOURS,180,'Good bake'),
('OVEN-2','BATCH-1002',current_timestamp()-INTERVAL 3 HOURS,current_timestamp()-INTERVAL 2 HOURS,175,'Even heat'),
('OVEN-1','BATCH-1003',current_timestamp()-INTERVAL 1 HOURS,current_timestamp(),200,'Overheated');

-- Waste Log
INSERT INTO waste_log VALUES
('WASTE-001','BATCH-1003','ING-001',1.5,'Burned batch',current_timestamp()),
('WASTE-002','BATCH-1003','ING-004',4,'Burned batch',current_timestamp());

-- Daily Production Summary
INSERT INTO daily_production VALUES
(current_date(),'PROD-001',2,240,1,1.5),
(current_date(),'PROD-002',1,80,0,0.0);


# 🧁 bake_goods Schema — Sample Data

In [0]:
%sql
USE CATALOG jim_bakery;
USE SCHEMA bake_goods;

-- Product Catalog
INSERT INTO product_catalog VALUES
('PROD-001','Blueberry Muffin','muffin','Fresh baked muffin with blueberries',3.50,true),
('PROD-002','Butter Croissant','pastry','Flaky croissant made with real butter',4.00,true),
('PROD-003','Chocolate Cake','cake','Rich chocolate layered cake',25.00,true);

-- SKU Metadata
INSERT INTO sku_metadata VALUES
('SKU-BM-S','PROD-001','small',0.15,'paper wrap'),
('SKU-BM-L','PROD-001','large',0.25,'paper wrap'),
('SKU-BC','PROD-002','standard',0.10,'paper sleeve'),
('SKU-CC-1','PROD-003','1lb',1.00,'box'),
('SKU-CC-2','PROD-003','2lb',2.00,'box');


# 👥 customers Schema — Sample Data

In [0]:
%sql
USE CATALOG jim_bakery;
USE SCHEMA customers;

-- Customer Profile
INSERT INTO customer_profile VALUES
('CUST-001','Alice Johnson','alice@example.com','555-1111','Gold',current_timestamp()),
('CUST-002','Brian Smith','brian@example.com','555-2222','Silver',current_timestamp()),
('CUST-003','Carla Gomez','carla@example.com','555-3333','Bronze',current_timestamp());

-- Orders
INSERT INTO orders VALUES
('ORD-1001','CUST-001',current_timestamp()-INTERVAL 1 DAY,7.00,'completed'),
('ORD-1002','CUST-002',current_timestamp()-INTERVAL 2 DAY,25.00,'completed'),
('ORD-1003','CUST-003',current_timestamp()-INTERVAL 3 DAY,3.50,'completed');

-- Order Items
INSERT INTO order_items VALUES
('ORD-1001','SKU-BM-S',2,3.50),
('ORD-1002','SKU-CC-1',1,25.00),
('ORD-1003','SKU-BM-L',1,3.50);
